#### Libraries and Packages to import

In [1]:
import os
import time
import random
import pandas as pd

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from tqdm import tqdm


#### File Paths (List of file paths)

In [2]:
INPUT_FILE = r"C:\Grainger VMI Items to Bid.xlsx"
OUTPUT_FILE = r"C\grainger_output.xlsx"

#### Back up for code Restart

In [ ]:
# Checking if there exists any file with same name and continue from where the code broke.

if os.path.exists(OUTPUT_FILE):
    df = pd.read_excel(OUTPUT_FILE)
    print("Resuming from existing output file.")
else:
    df = pd.read_excel(INPUT_FILE)
    print("Starting fresh from input file.")

# Ensure Product URL column exists
if "Product URL" not in df.columns:
    df["Product URL"] = ""

items = df["Material #"].astype(str).tolist()

#### Intitializing Browser

In [ ]:
driver = uc.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 25)

#### Helper Functions for Human like Approach

In [ ]:
# Helper functions to avoid page restriction due to multiple requests and mimic randomized human interaction.  

def human_pause(a=3, b=7):
    time.sleep(random.uniform(a, b))


def cooldown_pause():
    cooldown = random.uniform(120, 240)  # 2–4 minutes
    print(f"\nCooling down for {int(cooldown)} seconds...\n")
    time.sleep(cooldown)


#### Main Loop 

In [4]:
processed_since_break = 0

progress_bar = tqdm(range(len(items)), desc="Progress", unit="item")

for idx in progress_bar:

    item = items[idx]

    # Skip already-filled rows
    if (
        str(df.at[idx, "Brand"]).strip() not in ["", "nan"] and
        str(df.at[idx, "Mfr. Model"]).strip() not in ["", "nan"]
    ):
        progress_bar.set_postfix_str(f"Skipping {item}")
        continue

    # Slow navigation
    driver.get(f"https://www.grainger.com/search?searchBar=true&searchQuery={item}")
    human_pause(5, 10)

    # CAPTCHA detection
    if "captcha" in driver.current_url.lower():
        input("\nCAPTCHA detected. Solve it manually, then press ENTER here...")

    try:
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

        product_url = driver.current_url

        brand_elem = driver.find_element(
            By.XPATH, "//*[contains(text(),'Brand')]/ancestor::div[1]"
        )
        mfr_elem = driver.find_element(
            By.XPATH, "//*[contains(text(),'Mfr. Model')]/ancestor::div[1]"
        )

        brand = brand_elem.get_attribute("textContent").replace("Brand", "").strip()
        mfr_model = mfr_elem.get_attribute("textContent").replace("Mfr. Model", "").strip()

        # HARD STOP if data missing
        if not brand or not mfr_model:
            raise ValueError("Missing Brand or Mfr. Model")

    except Exception as e:
        print(f"\nDATA NOT FOUND for item {item}")
        print("Saving progress and stopping execution.")

        df.to_excel(OUTPUT_FILE, index=False)
        driver.quit()

        raise SystemExit("Stopped due to missing data (likely block)")

    # Write data
    df.at[idx, "Brand"] = brand
    df.at[idx, "Mfr. Model"] = mfr_model
    df.at[idx, "Product URL"] = product_url

    processed_since_break += 1

    progress_bar.set_postfix_str(
        f"Item {item} | Brand: {brand}"
    )

    # Save after every item (safe)
    df.to_excel(OUTPUT_FILE, index=False)

    # Human pacing
    human_pause(6, 12)

    # Cooldown after 18 processed items
    if processed_since_break >= 18:
        cooldown_pause()
        processed_since_break = 0


# CLEAN EXIT
df.to_excel(OUTPUT_FILE, index=False)
driver.quit()

print("\nCompleted successfully.")

Resuming from existing output file.


Progress:  63%|██████▎   | 450/714 [05:08<1:06:56, 15.22s/item, Item 1VAB5 | Brand: GRAINGER]              


Cooling down for 123 seconds...



Progress:  66%|██████▌   | 468/714 [12:37<1:15:10, 18.34s/item, Item 2KRF3 | Brand: GEORGIA-PACIFIC]


Cooling down for 156 seconds...



...


Cooling down for 195 seconds...



Progress:  98%|█████████▊| 702/714 [2:04:38<03:26, 17.23s/item, Item 35LV54 | Brand: MARTOR]        


Cooling down for 131 seconds...



Progress: 100%|██████████| 714/714 [2:10:19<00:00, 10.95s/item, Item 807CJ0 | Brand: LABELMASTER]



Completed successfully.
